<a href="https://colab.research.google.com/github/Guylord/Breast-Cancer-Classification-Scikit-learn-and-TensorFlow/blob/main/Breast_Cancer_Classification_Scikit_learn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importing relavant libraries

The code imports essential libraries for data manipulation, visualization, and machine learning. ***pandas*** and ***numpy*** handle data processing, while ***matplotlib*** and ***seaborn*** provide visualization tools. For model building, ***scikit-learn*** is used for algorithms, preprocessing, and evaluation. Additionally, tensorflow is imported for building and training neural network models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing modules
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder

# Machine learning algorithmns modules
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Model evaluation modules
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, log_loss

# Neural network modules
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Ucimlrepo package
The ucimlrepo package is used to load the Breast Cancer Wisconsin (Diagnostic) dataset from the UCI repository. The metadata display gives a comprehensive breakdown of the dataset's features, offering an essential understanding to guide the subsequent analysis steps.

In [ ]:
#Breast Cancer Wisconsin (Diagnostic) dataset- Metadata
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo
breast_cancer_wisconsin_diagnostic = fetch_ucirepo(id=17)
print(breast_cancer_wisconsin_diagnostic.metadata)

In [ ]:
# Dataset loading
df = pd.read_csv('https://archive.ics.uci.edu/static/public/17/data.csv')
df.info()

# Exploratory Data Analysis

In [ ]:
pd.set_option('display.max_columns', None)
df.head()

In [ ]:
#Distribution of the target variable
df['Diagnosis'].value_counts()

In [ ]:
#Distribution of the target variable (Visualization)
sns.countplot(x='Diagnosis', data=df)
plt.title('Distribution of Target Variable')
plt.xlabel('Target')
plt.ylabel('Count')
plt.show()

# Data preprocessing

In [ ]:
#Variables
X = df.drop(['ID','Diagnosis'], axis=1)
y = df['Diagnosis']

print('X shape: {}'.format(X.shape))
print('y shape: {}'.format(y.shape))

In [ ]:
#Target variable label encoding
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

random_state=925

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Model building (Scikit-learn)
The process entails training and assessing various machine learning models on the dataset. Several algorithms, including logistic regression, decision trees, and random forests, are evaluated to compare their accuracy, classification performance, and confusion matrices, helping determine the best model for the task.

In [ ]:
# Model evaluation function
def evaluate_model(model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    matrix = confusion_matrix(y_test, y_pred)
    return accuracy, report, matrix

In [ ]:
# Model building
models = {
    "Logistic Regression": LogisticRegression(),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Support Vector Machine": SVC(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier()}

results = {}
for model_name, model in models.items():
    accuracy, report, matrix = evaluate_model(model)
    results[model_name] = {
        "Accuracy": accuracy,
        "Classification Report": report,
        "Confusion Matrix": matrix
    }

for model_name, metrics in results.items():
    print(f"Model: {model_name}")
    print(f"Accuracy: {metrics['Accuracy']:.4f}")
    print("Classification Report:")
    print(metrics['Classification Report'])
    print("Confusion Matrix:")
    print(metrics['Confusion Matrix'])
    print("\n" + "="*60 + "\n")

# Fine-tuning
Logistic Regression and Random Forest models performed excellently with perfect accuracy. The Decision Tree, while good, has room for improvement. Therefore, the K-Nearest Neighbors, Support Vector Machine, and Decision Tree models are being fine-tuned with GridSearchCV to enhance their performance.

In [ ]:
# Fine-tuning function
def tune_model(model, param_grid):
    grid_search = GridSearchCV(estimator=model, param_grid=param_grid, scoring='accuracy', cv=5, n_jobs=-1)
    grid_search.fit(X_train, y_train)
    best_params = grid_search.best_params_
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    matrix = confusion_matrix(y_test, y_pred)
    return best_params, accuracy, report, matrix

In [ ]:
# Models and hyperparameters to be fine-tuned
models = {
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Support Vector Machine": SVC(),
    "Decision Tree": DecisionTreeClassifier()
  }

param_grids = {
    "K-Nearest Neighbors": {
        "n_neighbors": [3, 5, 7, 9],
        "metric": ["euclidean", "manhattan"]
    },
    "Support Vector Machine": {
        "C": [0.1, 1, 10, 100],
        "gamma": [1, 0.1, 0.01, 0.001],
        "kernel": ["linear", "rbf"]
    },
    "Decision Tree": {
        "criterion": ["gini", "entropy"],
        "max_depth": [None, 10, 20, 30, 40, 50],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4]
    }}

In [ ]:
# Tuning
best_params_results = {}
for model_name, model in models.items():
    print(f"Tuning {model_name}...")
    param_grid = param_grids[model_name]
    best_params, accuracy, report, matrix = tune_model(model, param_grid)
    best_params_results[model_name] = {
        "Best Parameters": best_params,
        "Accuracy": accuracy,
        "Classification Report": report,
        "Confusion Matrix": matrix
    }

In [ ]:
# Best results
for model_name, metrics in best_params_results.items():
    print(f"Model: {model_name}")
    print(f"Best Parameters: {metrics['Best Parameters']}")
    print(f"Accuracy: {metrics['Accuracy']:.4f}")
    print("Classification Report:")
    print(metrics['Classification Report'])
    print("Confusion Matrix:")
    print(metrics['Confusion Matrix'])
    print("\n" + "="*60 + "\n")

After fine-tuning, the models exhibit strong performance, with the Support Vector Machine achieving perfect accuracy (100%) and K-Nearest Neighbors improving to 97.37%. The Decision Tree model also saw significant improvement, reaching 99.12% accuracy. Each model's fine-tuning led to better precision, recall, and f1-scores, with SVM delivering flawless results across both classes, KNN showing slight improvements, and Decision Tree also classifying well with minimal errors.

# Building and training a Neural Network for binary classification

This code defines and trains a neural network for binary classification using Keras. The model consists of three hidden layers with ReLU activation and an output layer with a sigmoid activation. It is compiled with the Adam optimizer and binary cross-entropy loss, and trained for 10 epochs using a batch size of 32, with validation data provided to monitor performance.

In [ ]:
# Model building
model = Sequential()

model.add(Dense(128, input_dim=30, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])


history = model.fit(X_train, y_train,
                    validation_data=(X_test, y_test),
                    epochs=10,
                    batch_size=32)

# Model evaluation

In [ ]:
# Model evaluation
val_loss, val_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'Validation Loss: {val_loss:.4f}')
print(f'Validation Accuracy: {val_accuracy:.4f}')

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['loss'], label='Train Loss')
plt.title('Train accuracy vs. Train loss')
plt.xlabel('Epoch')
plt.ylabel(' ')
plt.legend(loc='lower right')

plt.tight_layout()
plt.show()

# Model fine-tuning
This code demonstrates the fine-tuning of a neural network for binary classification. The model utilizes the tanh activation function in the input and hidden layers, with a sigmoid activation in the output layer. Early stopping and learning rate reduction callbacks are employed to prevent overfitting and optimize training efficiency. The model is trained for 8 epochs with a batch size of 16, and the validation loss and accuracy are monitored and evaluated for performance.

In [ ]:
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout
model = Sequential()

# Input
model.add(Dense(128, input_dim=30, activation='tanh'))

# Hidden
model.add(Dense(64, activation='tanh'))
model.add(Dense(16, activation='tanh'))

# Output
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

early_stopping = EarlyStopping(monitor='val_loss', patience=1, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=5, min_lr=0.00001)
callbacks=[early_stopping, reduce_lr]

history = model.fit(X_train, y_train,
                    validation_data=(X_test, y_test),
                    epochs=8,
                    batch_size=16,
                    callbacks=callbacks)

# Model evaluation

In [ ]:
val_loss, val_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'Validation Loss: {val_loss:.4f}')
print(f'Validation Accuracy: {val_accuracy:.4f}')

In [ ]:
train_loss, train_accuracy = model.evaluate(X_train, y_train, verbose=0)
print(f'Train Loss: {train_loss:.4f}')
print(f'Train Accuracy: {train_accuracy:.4f}')

The model demonstrates strong performance, with a validation accuracy of 1.0000 and a low validation loss of 0.0297, indicating perfect classification on the test set. On the training set, the model maintains high accuracy at 98.90% with a train loss of 0.0538, suggesting effective learning without overfitting.